[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/main/notebooks/case_studies/research_assistant/plain_python.ipynb)

# Research assistant — plain Python

**Task.** Answer a user-supplied research question using only a bounded, safety-screened evidence catalogue.

**Read this notebook as:** choose a provider → define the question → plan → retrieve → screen evidence → extract and validate claims → synthesise → critique → report.


In [ ]:
import os
from getpass import getpass
from pathlib import Path

MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "research-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = os.getenv(
    "AGENTIC_TUTORIAL_LOCAL_MODEL_PATH",
    "models/local/Qwen3-0.6B-Q8_0.gguf",
)
API_MODEL_NAME = "gemini-3.1-flash-lite"
API_KEY_ENVIRONMENT_VARIABLE = "GEMINI_API_KEY"
SAVE_API_CREDENTIAL = True  # True stores it locally after the first hidden prompt
API_CREDENTIAL_FILE = Path(".private/gemini_api_key")
RESEARCH_QUESTION = "Which interventions reduce household food waste, and under what conditions?"
# Mock runtime is under one minute on CPU; local and API runs can be slower.

if MODEL_PROVIDER not in {"mock", "local", "api"}:
    raise ValueError("MODEL_PROVIDER must be mock, local or api")
if MODEL_PROVIDER == "local" and not LOCAL_MODEL_PATH:
    raise RuntimeError("Set AGENTIC_TUTORIAL_LOCAL_MODEL_PATH for local execution")
if MODEL_PROVIDER == "api":
    config_root = Path.cwd()
    while config_root.parent != config_root and not (config_root / "pyproject.toml").exists():
        config_root = config_root.parent
    credential_path = config_root / API_CREDENTIAL_FILE
    credential = os.getenv(API_KEY_ENVIRONMENT_VARIABLE, "")
    if not credential and credential_path.is_file():
        credential = credential_path.read_text(encoding="utf-8").strip()
    if not credential:
        credential = getpass(
            "Paste Gemini API key into the hidden input prompt, then press Enter: "
        ).strip()
    if not credential:
        raise RuntimeError(
            "No Gemini key was entered. Re-run this cell, use the hidden input prompt "
            "shown by Jupyter/VS Code, paste the key, and press Enter."
        )
    if SAVE_API_CREDENTIAL and not credential_path.is_file():
        credential_path.parent.mkdir(parents=True, exist_ok=True)
        credential_path.write_text(credential + "\n", encoding="utf-8")
        credential_path.chmod(0o600)
    os.environ[API_KEY_ENVIRONMENT_VARIABLE] = credential
    del credential

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic==2.12.5"], check=True)

In [ ]:
import json
from pathlib import Path
from typing import ClassVar, Literal

from pydantic import BaseModel, ConfigDict, Field

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.safety import (  # noqa: E402
    PolicyOutcome,
    SafetyEngine,
    SafetyPolicy,
    UntrustedContent,
)
from agentic_tutorial.schemas import CriticDecision, Message, MessageRole  # noqa: E402

catalogue = json.loads((ROOT / "data/research_assistant/evidence_catalogue.json").read_text())
fixture_path = ROOT / "data/research_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())

## Visible workflow

The eight functions below mirror the case-study skeleton used in the paper. The model proposes the plan, claims, synthesis and critique; Python performs bounded retrieval, untrusted-content screening, deterministic claim validation, budgeting, termination and reporting.


In [ ]:
# --- Declarations: typed outputs, model adapter, and shared workflow helpers ---
import re


class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class QueryPlan(Strict):
    schema_id: ClassVar[str] = "research.query.v1"
    queries: tuple[str, ...] = Field(min_length=1, max_length=4)


class Claim(Strict):
    source_id: str
    claim: str
    stance: Literal["supporting", "conflicting"]


class Ledger(Strict):
    schema_id: ClassVar[str] = "research.ledger.v1"
    claims: tuple[Claim, ...]


class Synthesis(Strict):
    schema_id: ClassVar[str] = "research.synthesis.v1"
    answer: str
    source_ids: tuple[str, ...]
    outcome: Literal["qualified_answer", "abstention"]


class CriticOutput(Strict):
    accepted: bool
    issues: tuple[str, ...] = ()


Claim.model_rebuild(_types_namespace={"Literal": Literal})
Ledger.model_rebuild(_types_namespace={"Claim": Claim})
Synthesis.model_rebuild(_types_namespace={"Literal": Literal})


def model():
    model_names = {
        "mock": MOCK_MODEL_NAME,
        "local": LOCAL_MODEL_NAME,
        "api": API_MODEL_NAME,
    }
    local_model_path = Path(LOCAL_MODEL_PATH) if LOCAL_MODEL_PATH else None
    if local_model_path is not None and not local_model_path.is_absolute():
        local_model_path = ROOT / local_model_path
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=local_model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


def search(query):
    terms = set(query.casefold().split())
    return [
        record
        for record in catalogue["records"]
        if terms & set((record["title"] + " " + record["passage"]).casefold().split())
    ]


async def propose(client, schema, prompt):
    response = await client.generate([user(prompt)], response_schema=schema)
    return schema.model_validate(response.structured_output)


def normalise_tokens(text):
    stopwords = {
        "a",
        "an",
        "and",
        "are",
        "as",
        "at",
        "be",
        "by",
        "for",
        "from",
        "in",
        "is",
        "it",
        "of",
        "on",
        "or",
        "that",
        "the",
        "to",
        "was",
        "were",
        "with",
    }
    return {token for token in re.findall(r"[a-z0-9]+", text.casefold()) if token not in stopwords}


def claim_is_grounded(claim, record, minimum_overlap=0.6):
    claim_tokens = normalise_tokens(claim.claim)
    passage_tokens = normalise_tokens(record["passage"])
    if len(claim_tokens) < 3:
        return False
    overlap = len(claim_tokens & passage_tokens) / len(claim_tokens)
    return overlap >= minimum_overlap

In [ ]:
# --- Conceptual case-study stages ---
async def plan(client, question):
    return await propose(
        client,
        QueryPlan,
        f"Create at most four focused catalogue-search queries that are sufficient to answer "
        f"this research question: {question!r}. Do not assume particular interventions in advance.",
    )


def retrieve(query_plan):
    return {record["source_id"]: record for query in query_plan.queries for record in search(query)}


def screen_evidence(records, safety):
    assessments = {
        source_id: safety.inspect_retrieved_content(
            UntrustedContent(source_id=source_id, text=record["passage"])
        )
        for source_id, record in records.items()
    }
    safe_records = {
        source_id: record
        for source_id, record in records.items()
        if assessments[source_id].decision.outcome in {PolicyOutcome.ALLOW, PolicyOutcome.TRANSFORM}
        and not assessments[source_id].indicators
    }
    isolated = [source_id for source_id, assessment in assessments.items() if assessment.indicators]
    return safe_records, isolated


async def extract_claims(client, safe_records):
    return await propose(
        client,
        Ledger,
        f"Extract one verbatim, source-grounded claim per record from {safe_records}. "
        "Preserve each source_id. Label reported reductions as supporting and inconsistent "
        "effects as conflicting.",
    )


def validate_claims(ledger, safe_records):
    return tuple(
        claim
        for claim in ledger.claims
        if claim.source_id in safe_records
        and claim_is_grounded(claim, safe_records[claim.source_id])
    )


async def synthesise(client, question, claims):
    return await propose(
        client,
        Synthesis,
        f"Answer {question!r} using only these validated claims: {claims}. "
        "State conditions and conflicts, cite source_ids, and use outcome qualified_answer.",
    )


async def critique(client, answer, safe_records):
    critic_output = await propose(
        client,
        CriticOutput,
        f"Accept only if this answer is supported, appropriately qualified and cited: "
        f"{answer.model_dump()}",
    )
    critic = CriticDecision(
        accepted=critic_output.accepted,
        issues=critic_output.issues,
    )
    citations_valid = set(answer.source_ids).issubset(safe_records)
    return critic, citations_valid


def report(
    *,
    question,
    query_plan,
    retrieved,
    safe_records,
    isolated,
    claims,
    answer=None,
    critic=None,
    citations_valid=False,
    model_calls,
):
    qualified = bool(
        answer is not None
        and critic is not None
        and critic.accepted
        and citations_valid
        and answer.outcome == "qualified_answer"
    )
    outcome = "qualified_answer" if qualified else "abstention"
    if not claims:
        termination = "no_validated_claims"
    elif not citations_valid:
        termination = "invalid_citations"
    elif critic is not None and not critic.accepted:
        termination = "critic_rejected"
    else:
        termination = "criteria_met"

    return {
        "question": question,
        "query_plan": query_plan.model_dump(),
        "outcome": outcome,
        "answer": None if answer is None else answer.model_dump(),
        "claim_ledger": [claim.model_dump() for claim in claims],
        "source_provenance": [] if answer is None else sorted(answer.source_ids),
        "execution_provenance": {
            "provider": MODEL_PROVIDER,
            "fixture_version": fixture["fixture_version"],
        },
        "trace": [
            {"event": "plan", "queries": query_plan.queries},
            {"event": "retrieve", "source_ids": sorted(retrieved)},
            {"event": "screen_evidence", "isolated": isolated},
            {
                "event": "extract_claims",
                "supporting": [c.source_id for c in claims if c.stance == "supporting"],
                "conflicting": [c.source_id for c in claims if c.stance == "conflicting"],
            },
            {"event": "validate_claims", "count": len(claims)},
            {"event": "synthesise", "completed": answer is not None},
            {
                "event": "critique",
                "accepted": None if critic is None else critic.accepted,
                "citations_valid": citations_valid,
            },
            {"event": "report", "outcome": outcome},
        ],
        "termination": termination,
        "usage": {"model_calls": model_calls, "tool_calls": len(query_plan.queries)},
        "safe_source_ids": sorted(safe_records),
    }


async def run_research(question=RESEARCH_QUESTION):
    client = model()
    safety = SafetyEngine(SafetyPolicy(allowed_tools=("catalogue_search",)))

    query_plan = await plan(client, question)
    retrieved = retrieve(query_plan)
    safe_records, isolated = screen_evidence(retrieved, safety)
    ledger = await extract_claims(client, safe_records)
    claims = validate_claims(ledger, safe_records)

    if not claims:
        return report(
            question=question,
            query_plan=query_plan,
            retrieved=retrieved,
            safe_records=safe_records,
            isolated=isolated,
            claims=claims,
            model_calls=2,
        )

    answer = await synthesise(client, question, claims)
    critic, citations_valid = await critique(client, answer, safe_records)
    return report(
        question=question,
        query_plan=query_plan,
        retrieved=retrieved,
        safe_records=safe_records,
        isolated=isolated,
        claims=claims,
        answer=answer,
        critic=critic,
        citations_valid=citations_valid,
        model_calls=4,
    )


# --- Execution: run the workflow and evaluate its observable result ---
first = await run_research(RESEARCH_QUESTION)
second = await run_research(RESEARCH_QUESTION) if MODEL_PROVIDER == "mock" else first
evaluation = {
    "component": len(first["claim_ledger"]) == 3,
    "trajectory": len(first["trace"]) == 8 and first["usage"]["model_calls"] <= 4,
    "task": first["outcome"] == "qualified_answer",
    "safety": "fw-004" not in first["source_provenance"],
    "repeated_run": first == second,
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), {"evaluation": evaluation, "result": first}
{
    "evaluation": evaluation,
    "result": first,
    "resource_report": first["usage"],
    "fallback": "abstain when no validated claims remain",
}

## Limitations

The tiny catalogue cannot establish external validity. Token-overlap validation is intentionally transparent and lightweight; stronger semantic entailment checks would be needed for open-domain evidence. Deterministic fixtures evaluate orchestration rather than real-model research quality.
